In [2]:
#pip installing all the libraries and getting import of all 

! pip install torch
! pip install tiktoken

In [3]:
# importing the tiktoken and enc then we will find n_vocab
import tiktoken

enc = tiktoken.get_encoding("gpt2")  #Loading the gpt2 tokenizer

print("Vocalbulary size: ",enc.n_vocab)

Vocalbulary size:  50257


In [5]:
#converting the text into integers

sentence = "I love death"
token_ids = enc.encode(sentence)

print("Sentence: ", sentence)
print("Token ID: ",token_ids)
print("How many tokens: ", len(token_ids))

Sentence:  I love death
Token ID:  [40, 1842, 1918]
How many tokens:  3


In [6]:
# Decoding back 
# converting the IDs into text
decoded = enc.decode(token_ids)
print("Decode: ", decoded)

Decode:  I love death


In [7]:
#see what individual token ID means
for id in token_ids:
    word = enc.decode([id])
    print(f"  ID  {id:6d}  -> '{word}'")

  ID      40  -> 'I'
  ID    1842  -> ' love'
  ID    1918  -> ' death'


In [8]:
#just testing with the long sentence

sentence2 = "Transformers are revolutionizing AI"
ids2 = enc.encode(sentence2)

print("\nSubword tokenization")

for id in ids2:
    print(f"  ID {id:6d} -> '{enc.decode([id])}'")



Subword tokenization
  ID  41762 -> 'Transform'
  ID    364 -> 'ers'
  ID    389 -> ' are'
  ID   5854 -> ' revolution'
  ID   2890 -> 'izing'
  ID   9552 -> ' AI'


#Exercise 2  ---- Token IDs ---> Embeddings

In [ ]:
#step1 - Import pytorch and define your config
import torch
import torch.nn as nn

vocab_size = enc.n_vocab  #50257 - one row per token
d_model = 64   # each token becomes 64 numbers

print("Vocab_size: ", vocab_size)
print("d_model: ",d_model)

Vocab_size:  50257
d_model:  64


In [10]:
#step2  ---- create the Embedding table

embedding_table = nn.Embedding(vocab_size,d_model)
print("Embedding table shape: ", embedding_table.weight.shape)

Embedding table shape:  torch.Size([50257, 64])


In [11]:
#step3 -- convert the tokenIDs into tensors 

token_ids = enc.encode("I love death")
token_ids_tensor = torch.tensor(token_ids)

print("Token_Id: ", token_ids)
print("Token Id tensor: ", token_ids_tensor)
print("Tensor shape: ",token_ids_tensor.shape)

Token_Id:  [40, 1842, 1918]
Token Id tensor:  tensor([  40, 1842, 1918])
Tensor shape:  torch.Size([3])


In [15]:
#step4 -- Look up the embeddings
embeddings = embedding_table(token_ids_tensor)

print("\nEmbedding shape:", embeddings.shape)

print("\nFirst token of 'I' embedding (first 8 numbers):")
print(embeddings[0, :8])

print("\nSecond token of 'love' embedding (first 8 numbers):")
print(embeddings[1, :8])

print("\nThird token of 'death' embedding (first 8 numbers):")
print(embeddings[2, :8])


Embedding shape: torch.Size([3, 64])

First token of 'I' embedding (first 8 numbers):
tensor([-0.9600,  1.0791,  0.0573, -0.9225, -0.9312, -0.7846,  0.9897,  1.1631],
       grad_fn=<SliceBackward0>)

Second token of 'love' embedding (first 8 numbers):
tensor([-0.7680, -0.8832, -1.6306,  0.0330, -0.5333, -0.6933,  0.6130,  0.6023],
       grad_fn=<SliceBackward0>)

Third token of 'death' embedding (first 8 numbers):
tensor([ 0.1025,  0.5407,  0.4246, -0.6527, -0.1244,  0.2492, -1.2820,  0.2576],
       grad_fn=<SliceBackward0>)


In [17]:
#step5 -- Add the batch dimension
# Real input always has a batch dimension
# Shape: [batch_size, seq_len]
batch = torch.tensor([
    enc.encode("I love cats too"),      # sentence 1
    enc.encode("cats love me too"), # sentence 2  ← must be same length!
])

print("Batch shape:", batch.shape)
# Expected: torch.Size([2, 4])
#                        ↑  ↑
#                    2 sentences, 4 tokens each

batch_embeddings = embedding_table(batch)
print("Batch embeddings shape:", batch_embeddings.shape)
# Expected: torch.Size([2, 4, 64])
#                        ↑  ↑   ↑
#                   batch seq  d_model

Batch shape: torch.Size([2, 4])
Batch embeddings shape: torch.Size([2, 4, 64])


#step 3 --- Adding positional Encoding


In [19]:
import math
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()

        # Build the PE matrix once — reuse forever
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)  # even dims
        pe[:, 1::2] = torch.cos(position * div_term)  # odd  dims

        # register_buffer = save pe inside model but don't train it
        # (positions are fixed math — not learned)
        self.register_buffer('pe', pe.unsqueeze(0))   # [1, max_len, d_model]

    def forward(self, x):
        # x shape: [batch, seq_len, d_model]
        seq_len = x.shape[1]

        # Slice PE to match actual sequence length and add
        return x + self.pe[:, :seq_len, :]


# ── Test the class ─────────────────────────────────────────────────────────
pos_enc = PositionalEncoding(d_model=64, max_len=100)

# Simulate a batch of 2 sentences, 4 tokens each
x   = torch.rand(2, 4, 64)       # [batch=2, seq=4, d_model=64]
out = pos_enc(x)

print("\n=== PositionalEncoding class test ===")
print("Input  shape:", x.shape)    # [2, 4, 64]
print("Output shape:", out.shape)  # [2, 4, 64]  ← same shape ✓
print("\nAre input and output different?", not torch.equal(x, out))



=== PositionalEncoding class test ===
Input  shape: torch.Size([2, 4, 64])
Output shape: torch.Size([2, 4, 64])

Are input and output different? True


#self attention from scratch

In [23]:
import torch.nn.functional as F
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        self.d_k = d_k

        # Three projection matrices
        self.W_q = nn.Linear(d_model, d_k, bias=False)
        self.W_k = nn.Linear(d_model, d_k, bias=False)
        self.W_v = nn.Linear(d_model, d_k, bias=False)

    def forward(self, x):
        # x: [batch, seq_len, d_model]

        # STEP 1 — Project to Q, K, V
        Q = self.W_q(x)                              # [batch, seq, d_k]
        K = self.W_k(x)                              # [batch, seq, d_k]
        V = self.W_v(x)                              # [batch, seq, d_k]

        # STEP 2 — Compute scores
        scores = torch.matmul(Q, K.transpose(-2,-1)) # [batch, seq, seq]

        # STEP 3 — Scale
        scores = scores / math.sqrt(self.d_k)

        # STEP 4 — Softmax
        weights = F.softmax(scores, dim=-1)          # [batch, seq, seq]

        # STEP 5 — Weighted blend of V
        output = torch.matmul(weights, V)            # [batch, seq, d_k]

        return output, weights   # return weights too so we can inspect them


# ── Dry Run ────────────────────────────────────────────────────────────────
torch.manual_seed(42)
attn = SelfAttention(d_model=8, d_k=8)
x    = torch.rand(1, 3, 8)   # batch=1, 3 words, 8 dims

output, weights = attn(x)

print("\n=== SELF ATTENTION CLASS TEST ===")
print("Input  shape :", x.shape)        # [1, 3, 8]
print("Output shape :", output.shape)   # [1, 3, 8] ✓
print("\nAttention weights (who attends to whom):")
print(weights.squeeze())
print("\nRow sums (must be 1.0):")
print(weights.squeeze().sum(dim=-1))


=== SELF ATTENTION CLASS TEST ===
Input  shape : torch.Size([1, 3, 8])
Output shape : torch.Size([1, 3, 8])

Attention weights (who attends to whom):
tensor([[0.3551, 0.2930, 0.3519],
        [0.3569, 0.2853, 0.3578],
        [0.3476, 0.2940, 0.3584]], grad_fn=<SqueezeBackward0>)

Row sums (must be 1.0):
tensor([1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [24]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        # d_model must divide evenly into heads
        assert d_model % num_heads == 0, \
            f"d_model {d_model} must be divisible by num_heads {num_heads}"

        self.d_model    = d_model
        self.num_heads  = num_heads
        self.d_k        = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):
        # x: [batch, seq, d_model]
        # → [batch, seq, heads, d_k]
        # → [batch, heads, seq, d_k]
        batch, seq, _ = x.shape
        x = x.view(batch, seq, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, x):
        batch, seq, _ = x.shape

        # STEP 1 — Project
        Q = self.split_heads(self.W_q(x))   # [batch, heads, seq, d_k]
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        # STEP 2 — Attend
        scores  = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        weights = F.softmax(scores, dim=-1)
        attn    = torch.matmul(weights, V)   # [batch, heads, seq, d_k]

        # STEP 3 — Concat
        attn = attn.transpose(1,2).contiguous()
        attn = attn.view(batch, seq, self.d_model)  # [batch, seq, d_model]

        # STEP 4 — Mix heads
        return self.W_o(attn)


# ── Dry Run ────────────────────────────────────────────────────────────────
torch.manual_seed(42)
mha = MultiHeadAttention(d_model=8, num_heads=2)
x   = torch.rand(1, 3, 8)
out = mha(x)

print("\n=== MULTI HEAD ATTENTION TEST ===")
print(f"Input  shape : {x.shape}")     # [1, 3, 8]
print(f"Output shape : {out.shape}")   # [1, 3, 8] ✓
print(f"\nParameters in MHA:")
total = sum(p.numel() for p in mha.parameters())
print(f"  Total: {total}")             # 4 × 8×8 = 256 parameters




=== MULTI HEAD ATTENTION TEST ===
Input  shape : torch.Size([1, 3, 8])
Output shape : torch.Size([1, 3, 8])

Parameters in MHA:
  Total: 256
